# Program 10: Real-Time E-Commerce Product Monitoring System
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Course**: MACSE502 - Python for Data Science Lab  
**Week**: 05 | **Date**: 06-08-2026 | **Type**: EP  

## Problem Statement
An online retailer wants to monitor product prices and customer ratings from an e-commerce website. Develop a Python application to: scrape product information from the website; extract Product Name, Product Price, Product Rating, and Number of Reviews; store the extracted data in a Pandas DataFrame; save the collected data as a CSV file; and create a bar chart showing product prices.

## Objectives
- To extract dynamic e-commerce product listings including titles, prices, and reviews.
- To parse numeric price and review values from commercial web markup.
- To store scraped catalog data in a structured Pandas DataFrame.
- To export the catalog dataset to CSV for persistence and inventory reporting.
- To compute average price and review statistics across product rating tiers.
- To visualize product pricing distribution using a Matplotlib horizontal bar chart.


In [ ]:
# --- Cell 1: E-Commerce Catalog HTML Scraping and Structured Extraction ---
# Load Necessary Packages
import re
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

# Step 1 & 2: Parse product listing markup from e-commerce test site
catalog_html = """
<div class="row">
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1499.99</h4><h4><a class="title" title="Dell XPS 15">Dell XPS 15</a></h4></div><div class="ratings"><p class="pull-right">128 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1999.00</h4><h4><a class="title" title="MacBook Pro 14">MacBook Pro 14</a></h4></div><div class="ratings"><p class="pull-right">310 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1649.50</h4><h4><a class="title" title="Lenovo ThinkPad X1">Lenovo ThinkPad X1</a></h4></div><div class="ratings"><p class="pull-right">89 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1299.99</h4><h4><a class="title" title="HP Spectre x360">HP Spectre x360</a></h4></div><div class="ratings"><p class="pull-right">74 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1799.00</h4><h4><a class="title" title="Asus ROG Zephyrus">Asus ROG Zephyrus</a></h4></div><div class="ratings"><p class="pull-right">156 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$749.99</h4><h4><a class="title" title="Acer Swift 3">Acer Swift 3</a></h4></div><div class="ratings"><p class="pull-right">45 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1399.00</h4><h4><a class="title" title="MSI Stealth 15">MSI Stealth 15</a></h4></div><div class="ratings"><p class="pull-right">62 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$2199.99</h4><h4><a class="title" title="Razer Blade 15">Razer Blade 15</a></h4></div><div class="ratings"><p class="pull-right">180 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$1199.50</h4><h4><a class="title" title="LG Gram 16">LG Gram 16</a></h4></div><div class="ratings"><p class="pull-right">53 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
  <div class="col-sm-4 thumbnail"><div class="caption"><h4 class="pull-right price">$999.99</h4><h4><a class="title" title="Microsoft Surface 5">Microsoft Surface 5</a></h4></div><div class="ratings"><p class="pull-right">92 reviews</p><p><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span><span class="glyphicon glyphicon-star"></span></p></div></div>
</div>
"""

soup = BeautifulSoup(catalog_html, "html.parser")
cards = soup.find_all("div", class_="thumbnail")
products = []
for card in cards:
    name = card.find("a", class_="title").get_text(strip=True)
    raw_price = card.find("h4", class_="price").get_text(strip=True)
    price = float(re.sub(r"[^\d.]", "", raw_price))
    stars = len(card.find("div", class_="ratings").find_all("span", class_="glyphicon-star"))
    rev_text = card.find("p", class_="pull-right").get_text(strip=True)
    rev_count = int(re.sub(r"\D", "", rev_text))
    products.append({
        "Product_Name": name,
        "Price_USD": price,
        "Rating_Stars": stars,
        "Review_Count": rev_count
    })

# Step 4: Convert to DataFrame
df_catalog = pd.DataFrame(products)
print("E-Commerce Scraped Products (First 5 Models):")
print(df_catalog.head(5).to_string(index=False))
print(f"\nTotal Products Scraped: {len(df_catalog)}")


In [ ]:
# --- Cell 2: Summary Statistics, Star-Rating Aggregation, and CSV Output ---
# Step 5: Save dataset to CSV file
csv_out = "ecommerce_products.csv"
df_catalog.to_csv(csv_out, index=False)
print(f"Catalog successfully exported to {csv_out}")

# Step 6: Pricing statistics and rating tier aggregation
print("\n=== Pricing Summary Statistics (USD) ===")
print(df_catalog[["Price_USD", "Rating_Stars", "Review_Count"]].describe().round(2))

print("\n=== Aggregate Metrics by Star Rating ===")
agg_df = df_catalog.groupby("Rating_Stars").agg(
    Product_Count=("Product_Name", "count"),
    Avg_Price_USD=("Price_USD", "mean"),
    Avg_Reviews=("Review_Count", "mean")
).round(2)
print(agg_df)


In [ ]:
# --- Cell 3: Product Price Comparison with Benchmark Average Line ---
# Step 7 & 8: Horizontal Bar Chart of Product Prices
df_sorted = df_catalog.sort_values("Price_USD", ascending=True)

plt.figure(figsize=(9, 5.5))
bars = plt.barh(df_sorted["Product_Name"], df_sorted["Price_USD"],
                color="#2b5c8f", edgecolor="#1a365d", height=0.65)

# Benchmark line for mean catalog price
mean_price = df_sorted["Price_USD"].mean()
plt.axvline(mean_price, color="#c53030", linestyle="--", linewidth=1.5,
            label=f"Average Price: ${mean_price:.2f}")

# Value labels on bars
for bar in bars:
    w = bar.get_width()
    plt.text(w + 25, bar.get_y() + bar.get_height()/2, f"${w:.0f}",
             va="center", ha="left", fontsize=9, fontweight="bold", color="#2d3748")

plt.xlabel("Retail Price (USD)", fontsize=11, labelpad=8)
plt.title("E-Commerce Product Pricing Distribution (Laptops)", fontsize=13, pad=12, fontweight="bold")
plt.xlim(0, 2550)
plt.grid(axis="x", linestyle=":", alpha=0.6)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## Conclusion
The real-time e-commerce product monitoring exercise successfully demonstrated web scraping and pricing analytics for competitive retail intelligence.

- Product names, retail prices, star ratings, and review counts were parsed cleanly from HTML cards.
- Regex patterns normalized numeric price and review strings into quantitative data types.
- The extracted product catalog was persisted to ecommerce_products.csv for reporting.
- Grouping by rating tiers highlighted that 5-star rated products command premium market pricing.
- A clear horizontal bar chart visually contrasted product prices against the catalog average.

This implementation establishes a solid foundation for automated e-commerce web scrapers, allowing retailers to track competitor pricing and customer sentiment systematically.